<a href="https://colab.research.google.com/github/gugrajah/data-loading-with-python-and-ai/blob/main/data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Engineering for ETL: Data Ingestion

#### Outcomes
🎯 1. Extracting Data from APIs & Handling API Challenges – Work with REST APIs, authentication, rate limits, retries, and pagination to extract data efficiently.

🎯 2. Schema Management & Automatic Normalization – Use dlt to infer schemas, flatten nested JSON, extract lists into child tables, and handle schema evolution automatically.

🎯 3. Incremental Data Extraction & State Tracking – Load only new or modified records, avoiding unnecessary reprocessing and improving pipeline efficiency.

🎯 4. Loading Data into Various Destinations – Store data in DuckDB, BigQuery, Snowflake, or a Data Lake while ensuring efficient schema mapping and performance.

🎯 5. Automating & Orchestrating Pipelines – Deploy, schedule, and maintain ingestion workflows with Dagster, GitHub Actions, and Cron Jobs.

🎯 6. Scaling Data Pipelines Efficiently – Handle large-scale data ingestion while optimizing performance, retries, and parallel execution.

#### 01. API data extraction: Batch


In [ ]:
import requests

url = 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp/events'

response = requests.get(url)
response.json()

[{'id': '17072537694',
  'type': 'WatchEvent',
  'actor': {'id': 66855368,
   'login': 'frankvier',
   'display_login': 'frankvier',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/frankvier',
   'avatar_url': 'https://avatars.githubusercontent.com/u/66855368?'},
  'repo': {'id': 419661684,
   'name': 'DataTalksClub/data-engineering-zoomcamp',
   'url': 'https://api.github.com/repos/DataTalksClub/data-engineering-zoomcamp'},
  'payload': {'action': 'started'},
  'public': True,
  'created_at': '2026-10-13T06:28:07Z',
  'org': {'id': 72699292,
   'login': 'DataTalksClub',
   'gravatar_id': '',
   'url': 'https://api.github.com/orgs/DataTalksClub',
   'avatar_url': 'https://avatars.githubusercontent.com/u/72699292?'}},
 {'id': '17067659841',
  'type': 'WatchEvent',
  'actor': {'id': 341789624,
   'login': 'JayPatidar0601',
   'display_login': 'JayPatidar0601',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/JayPatidar0601',
   'avatar_url': 'https://avatars.gi

* rate limits

In [ ]:
# monitor rate limits
url = 'https://api.github.com/rate_limit'

requests.get(url).json()

{'resources': {'code_search': {'limit': 60,
   'remaining': 59,
   'reset': 1791878130,
   'used': 1},
  'core': {'limit': 60, 'remaining': 59, 'reset': 1791878130, 'used': 1},
  'graphql': {'limit': 0, 'remaining': 0, 'reset': 1791878131, 'used': 0},
  'integration_manifest': {'limit': 5000,
   'remaining': 5000,
   'reset': 1791878131,
   'used': 0},
  'search': {'limit': 10, 'remaining': 10, 'reset': 1791874591, 'used': 0}},
 'rate': {'limit': 60, 'remaining': 59, 'reset': 1791878130, 'used': 1}}

In [ ]:
# pause requests and implement retries
import time

remaining_requests = requests.get(url).json()['rate']['remaining']

if remaining_requests == 0:
    time.sleep(30)


* Authentication

In [ ]:
url = 'https://api.github.com/user'

requests.get(url).json()

{'message': 'Requires authentication',
 'documentation_url': 'https://docs.github.com/rest',
 'status': '401'}

In [ ]:
import os
import requests

# 1. Fetch token depending on environment
try:
    from google.colab import userdata
    API_TOKEN = userdata.get('ACCESS_TOKEN')
except (ImportError, ModuleNotFoundError):
    from dotenv import load_dotenv
    load_dotenv('access.env')
    API_TOKEN = os.getenv('ACCESS_TOKEN')

# 2. Make authenticated request
headers = {'Authorization': f'Bearer {API_TOKEN}'}
url = 'https://api.github.com/user'
response = requests.get(url, headers=headers)
response.json()

